数据清洗 yellow_taxi_data_cleaned

In [ ]:
import duckdb
import os

conn = duckdb.connect()
conn.execute("SET memory_limit = '24GB'")

# 1. 查看数据目录结构，确认能读到文件
data_dir = "D:/yellow_taxi/yellow_taxi_data/"
parquet_files = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith('.parquet')]
print(f"找到 {len(parquet_files)} 个 parquet 文件")
# 查看其中一个文件的 schema
sample_file = parquet_files[0]
schema_df = conn.execute(f"SELECT * FROM '{sample_file}' LIMIT 0").fetchdf()
print("字段列表：", list(schema_df.columns))

# 2. 用SQL定义清洗逻辑
cleaning_sql = """
CREATE OR REPLACE VIEW cleaned_trips AS
SELECT 
    VendorID,
    tpep_pickup_datetime,
    tpep_dropoff_datetime,
    passenger_count,
    trip_distance,
    PULocationID,
    DOLocationID,
    payment_type,
    fare_amount,
    extra,
    mta_tax,
    tip_amount,
    tolls_amount,
    total_amount,
    -- 计算行程时长（分钟）
    (EPOCH(tpep_dropoff_datetime - tpep_pickup_datetime) / 60) AS trip_duration_minutes
FROM read_parquet('D:/yellow_taxi/yellow_taxi_data/*.parquet')
WHERE 
    passenger_count > 0
    AND trip_distance > 0
    AND fare_amount > 0
    AND total_amount > 0
    AND tpep_dropoff_datetime > tpep_pickup_datetime
    AND PULocationID IS NOT NULL
    AND DOLocationID IS NOT NULL
    AND EXTRACT(YEAR FROM tpep_pickup_datetime) BETWEEN 2021 AND 2026
    -- 行程时长
    AND EPOCH(tpep_dropoff_datetime - tpep_pickup_datetime) > 60   -- 至少 60 秒
    AND EPOCH(tpep_dropoff_datetime - tpep_pickup_datetime) < 86400 -- 不超过 24 小时
    -- 车费单价每英里 0.5~100 美元之间
    AND (total_amount / NULLIF(trip_distance, 0)) BETWEEN 0.5 AND 100
"""
conn.execute(cleaning_sql)

# 3. 验证清洗效果
count_before = conn.execute("SELECT COUNT(*) FROM read_parquet('D:/yellow_taxi/yellow_taxi_data/*.parquet')").fetchone()[0]
count_after = conn.execute("SELECT COUNT(*) FROM cleaned_trips").fetchone()[0]
print(f"清洗前记录数：{count_before:,}")
print(f"清洗后记录数：{count_after:,}")
print(f"清洗后保留比例：{count_after/count_before*100:.2f}%")
sample_data = conn.execute("SELECT * FROM cleaned_trips LIMIT 10").fetchdf()
print("清洗后数据样例：")
print(sample_data)

# 4. 按年份分区写入清洗后的 Parquet 文件
conn.execute("""
COPY (
    SELECT *, (EXTRACT(YEAR FROM tpep_pickup_datetime)::INTEGER)
    AS pickup_year FROM cleaned_trips
) TO 'D:/yellow_taxi/yellow_taxi_data_cleaned/' 
(FORMAT PARQUET, PARTITION_BY (pickup_year), OVERWRITE_OR_IGNORE)
""")
print("清洗后的数据已写入 D:/yellow_taxi/yellow_taxi_data_cleaned/ 目录")

conn.close()

找到 63 个 parquet 文件
字段列表： ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee']
清洗前记录数：209,840,160
清洗后记录数：178,170,956
清洗后保留比例：84.91%
清洗后数据样例：
   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         1  2021-01-01 00:30:10   2021-01-01 00:36:12              1.0   
1         1  2021-01-01 00:43:30   2021-01-01 01:11:06              1.0   
2         2  2021-01-01 00:31:49   2021-01-01 00:48:21              1.0   
3         1  2021-01-01 00:16:29   2021-01-01 00:24:30              1.0   
4         1  2021-01-01 00:00:28   2021-01-01 00:17:28              1.0   
5         1  2021-01-01 00:12:29   2021-01-01 00:30:34              1.0   
6         1  2021-01-01 00:39:16   2021-01-01 01:00:13       